# XGBoost

### Import needed libraries

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, log_loss, accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold


### Initialize dataframes with aggregate data

In [2]:
phy30sdf = pd.read_csv('../../exports/data/phys_agg_30s.csv', encoding="utf-8-sig")
phy16sdf = pd.read_csv('../../exports/data/phys_agg_16s.csv', encoding="utf-8-sig")
phy10sdf = pd.read_csv('../../exports/data/phys_agg_10s.csv', encoding="utf-8-sig")

scada30sdf = pd.read_csv('../../exports/data/scada_resolved_agg_30s.csv', encoding="utf-8-sig")
scada16sdf = pd.read_csv('../../exports/data/scada_resolved_agg_16s.csv', encoding="utf-8-sig")
scada10sdf = pd.read_csv('../../exports/data/scada_resolved_agg_10s.csv', encoding="utf-8-sig")

# drop min_value and max_value columns from all dataframes
for df in [phy30sdf, phy16sdf, phy10sdf, scada30sdf, scada16sdf, scada10sdf]:
    df = df.drop(['min_value', 'max_value'], axis=1, errors='ignore', inplace=True)


## Some quick data analysis

#### Physical data

In [3]:

# break up rows with multiple attacks
# For the 30 second bucket size (physical)
phy30sdf['attack_types'] = phy30sdf['attack_types'].apply(lambda x: np.array(eval(x)) if pd.notnull(x) else np.array([]))
phy30sdf = phy30sdf.explode('attack_types').reset_index(drop=True)
phy30sdf['attack_types'] = phy30sdf['attack_types'].apply(lambda x: x if x != 'nomal' else 'normal')

In [4]:
# look at feature info
phy30sdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17480 entries, 0 to 17479
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   bucket            17480 non-null  object 
 1   system_id         17480 non-null  object 
 2   prop_key          17480 non-null  object 
 3   asset_id          17480 non-null  object 
 4   avg_value         17480 non-null  float64
 5   num_measurements  17480 non-null  int64  
 6   num_attacks       17480 non-null  int64  
 7   attack_types      17480 non-null  object 
dtypes: float64(1), int64(2), object(5)
memory usage: 1.1+ MB


In [5]:
# The percent of the data that is an attack
print(f"phy30sdf: {(phy30sdf['num_attacks']>0).mean()*100:.2f}% rows are attacks")

phy30sdf: 40.05% rows are attacks


In [6]:
phy16sdf['attack_types'] = phy16sdf['attack_types'].apply(lambda x: np.array(eval(x)) if pd.notnull(x) else np.array([]))
phy16sdf = phy16sdf.explode('attack_types').reset_index(drop=True)
phy16sdf['attack_types'] = phy16sdf['attack_types'].apply(lambda x: x if x != 'nomal' else 'normal')

In [7]:
# look at feature info
phy16sdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30120 entries, 0 to 30119
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   bucket            30120 non-null  object 
 1   system_id         30120 non-null  object 
 2   prop_key          30120 non-null  object 
 3   asset_id          30120 non-null  object 
 4   avg_value         30120 non-null  float64
 5   num_measurements  30120 non-null  int64  
 6   num_attacks       30120 non-null  int64  
 7   attack_types      30120 non-null  object 
dtypes: float64(1), int64(2), object(5)
memory usage: 1.8+ MB


In [8]:
# The percent of the data that is an attack
print(f"phy16sdf: {(phy16sdf['num_attacks']>0).mean()*100:.2f}% rows are attacks")

phy30sdf: 30.15% rows are attacks


In [9]:
# break up rows with multiple attacks
# For the 10 second bucket size (physical)
phy10sdf['attack_types'] = phy10sdf['attack_types'].apply(lambda x: np.array(eval(x)) if pd.notnull(x) else np.array([]))
phy10sdf = phy10sdf.explode('attack_types').reset_index(drop=True)
phy10sdf['attack_types'] = phy10sdf['attack_types'].apply(lambda x: x if x != 'nomal' else 'normal')

In [10]:
# look at feature info
phy10sdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46480 entries, 0 to 46479
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   bucket            46480 non-null  object 
 1   system_id         46480 non-null  object 
 2   prop_key          46480 non-null  object 
 3   asset_id          46480 non-null  object 
 4   avg_value         46480 non-null  float64
 5   num_measurements  46480 non-null  int64  
 6   num_attacks       46480 non-null  int64  
 7   attack_types      46480 non-null  object 
dtypes: float64(1), int64(2), object(5)
memory usage: 2.8+ MB


In [11]:
# The percentage of the data that is attack
print(f"phy10sdf: {(phy10sdf['num_attacks']>0).mean()*100:.2f}% rows are attacks")

phy10sdf: 26.33% rows are attacks


In [12]:
# break up rows with multiple attacks
# For the 30 second bucket size (network)
scada30sdf['attack_types'] = scada30sdf['attack_types'].apply(lambda x: np.array(eval(x)) if pd.notnull(x) else np.array([]))
scada30sdf = scada30sdf.explode('attack_types').reset_index(drop=True)
scada30sdf['attack_types'] = scada30sdf['attack_types'].apply(lambda x: x if x != 'nomal' else 'normal')

In [13]:
# look at feature info
scada30sdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 352738 entries, 0 to 352737
Data columns (total 21 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   bucket                     352738 non-null  object 
 1   system_id                  352738 non-null  object 
 2   protocol                   352738 non-null  object 
 3   avg_size                   352738 non-null  float64
 4   source_total_packets       352738 non-null  int64  
 5   destination_total_packets  352738 non-null  int64  
 6   min_size                   352738 non-null  int64  
 7   max_size                   352738 non-null  int64  
 8   num_connections            352738 non-null  int64  
 9   source_ip                  352439 non-null  object 
 10  source_port                352352 non-null  float64
 11  source_mac                 352738 non-null  object 
 12  destination_ip             352439 non-null  object 
 13  destination_port           35

In [14]:
# The percentage of the data that is attack
print(f"scada30sdf: {(scada30sdf['num_attacks']>0).mean()*100:.2f}% rows are attacks")

scada30sdf: 54.23% rows are attacks


In [15]:
# break up rows with multiple attacks
# For the 16 second bucket size (network)
scada16sdf['attack_types'] = scada16sdf['attack_types'].apply(lambda x: np.array(eval(x)) if pd.notnull(x) else np.array([]))
scada16sdf = scada30sdf.explode('attack_types').reset_index(drop=True)
scada16sdf['attack_types'] = scada16sdf['attack_types'].apply(lambda x: x if x != 'nomal' else 'normal')

In [16]:
# look at feature info
scada16sdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 352738 entries, 0 to 352737
Data columns (total 21 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   bucket                     352738 non-null  object 
 1   system_id                  352738 non-null  object 
 2   protocol                   352738 non-null  object 
 3   avg_size                   352738 non-null  float64
 4   source_total_packets       352738 non-null  int64  
 5   destination_total_packets  352738 non-null  int64  
 6   min_size                   352738 non-null  int64  
 7   max_size                   352738 non-null  int64  
 8   num_connections            352738 non-null  int64  
 9   source_ip                  352439 non-null  object 
 10  source_port                352352 non-null  float64
 11  source_mac                 352738 non-null  object 
 12  destination_ip             352439 non-null  object 
 13  destination_port           35

In [17]:
# The percentage of the data that is attack
print(f"scada16sdf: {(scada16sdf['num_attacks']>0).mean()*100:.2f}% rows are attacks")

scada16sdf: 54.23% rows are attacks


In [18]:
# break up rows with multiple attacks
# For the 10 second bucket size (network)
scada10sdf['attack_types'] = scada10sdf['attack_types'].apply(lambda x: np.array(eval(x)) if pd.notnull(x) else np.array([]))
scada10sdf = scada30sdf.explode('attack_types').reset_index(drop=True)
scada10sdf['attack_types'] = scada10sdf['attack_types'].apply(lambda x: x if x != 'nomal' else 'normal')

In [19]:
# look at feature info
scada10sdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 352738 entries, 0 to 352737
Data columns (total 21 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   bucket                     352738 non-null  object 
 1   system_id                  352738 non-null  object 
 2   protocol                   352738 non-null  object 
 3   avg_size                   352738 non-null  float64
 4   source_total_packets       352738 non-null  int64  
 5   destination_total_packets  352738 non-null  int64  
 6   min_size                   352738 non-null  int64  
 7   max_size                   352738 non-null  int64  
 8   num_connections            352738 non-null  int64  
 9   source_ip                  352439 non-null  object 
 10  source_port                352352 non-null  float64
 11  source_mac                 352738 non-null  object 
 12  destination_ip             352439 non-null  object 
 13  destination_port           35

In [20]:
# The percentage of the data that is attack
print(f"scada10sdf: {(scada10sdf['num_attacks']>0).mean()*100:.2f}% rows are attacks")

scada10sdf: 54.23% rows are attacks


## Balance the datasets

In [21]:
# percent of normal vs attack rows
def print_attack_normal_ratio(df, name):
    total_rows = len(df)
    normal_rows = len(df[df['attack_types'] == 'normal'])
    attack_rows = total_rows - normal_rows
    print(f"[name] : {normal_rows} normal_rows ({(normal_rows/total_rows)*100:.2f}%), {attack_rows} attack rows ({(attack_rows/total_rows)*100:.2f}%)")

### Balance phydf

In [22]:
'''# original vs normal rows
total_rows = len(phy30sdf)
normal_rows = len(phy30sdf[phy30sdf['num_attacks'] == 0])
attack_rows = total_rows - normal_rows
print(f"[name] : {normal_rows} normal_rows ({(normal_rows/total_rows)*100:.2f}%), {attack_rows} attack rows ({(attack_rows/total_rows)*100:.2f}%)")

# balance the phydf dataframe
normal_rows = phy30sdf[phy30sdf['num_attacks'] == 0]
attack_rows = phy30sdf[phy30sdf['num_attacks'] == 1]
num_normal = len(normal_rows)
num_attack = len(attack_rows)

normal_rows = normal_rows.sample(n=num_attack, random_state=42)

phys30df_balanced = pd.concat([normal_rows, attack_rows]).reset_index(drop=True)

# new attack vs normal ratio
total_rows = len(phys30df_balanced)
normal_rows = len(phys30df_balanced[phys30df_balanced['num_attacks'] == 0])
attack_rows = total_rows - normal_rows
print(f"[name] : {normal_rows} normal_rows ({(normal_rows/total_rows)*100:.2f}%), {attack_rows} attack rows ({(attack_rows/total_rows)*100:.2f}%)")
'''

'# original vs normal rows\ntotal_rows = len(phy30sdf)\nnormal_rows = len(phy30sdf[phy30sdf[\'num_attacks\'] == 0])\nattack_rows = total_rows - normal_rows\nprint(f"[name] : {normal_rows} normal_rows ({(normal_rows/total_rows)*100:.2f}%), {attack_rows} attack rows ({(attack_rows/total_rows)*100:.2f}%)")\n\n# balance the phydf dataframe\nnormal_rows = phy30sdf[phy30sdf[\'num_attacks\'] == 0]\nattack_rows = phy30sdf[phy30sdf[\'num_attacks\'] == 1]\nnum_normal = len(normal_rows)\nnum_attack = len(attack_rows)\n\nnormal_rows = normal_rows.sample(n=num_attack, random_state=42)\n\nphys30df_balanced = pd.concat([normal_rows, attack_rows]).reset_index(drop=True)\n\n# new attack vs normal ratio\ntotal_rows = len(phys30df_balanced)\nnormal_rows = len(phys30df_balanced[phys30df_balanced[\'num_attacks\'] == 0])\nattack_rows = total_rows - normal_rows\nprint(f"[name] : {normal_rows} normal_rows ({(normal_rows/total_rows)*100:.2f}%), {attack_rows} attack rows ({(attack_rows/total_rows)*100:.2f}%)")\

In [23]:
'''# original vs normal rows
total_rows = len(phy16sdf)
normal_rows = len(phy16sdf[phy16sdf['num_attacks'] == 0])
attack_rows = total_rows - normal_rows
print(f"[name] : {normal_rows} normal_rows ({(normal_rows/total_rows)*100:.2f}%), {attack_rows} attack rows ({(attack_rows/total_rows)*100:.2f}%)")

# balance the phydf dataframe
normal_rows = phy16sdf[phy16sdf['num_attacks'] == 0]
attack_rows = phy16sdf[phy16sdf['num_attacks'] == 1]
num_normal = len(normal_rows)
num_attack = len(attack_rows)

normal_rows = normal_rows.sample(n=num_attack, random_state=42)

phys16df_balanced = pd.concat([normal_rows, attack_rows]).reset_index(drop=True)

# new attack vs normal ratio
total_rows = len(phys16df_balanced)
normal_rows = len(phys16df_balanced[phys16df_balanced['num_attacks'] == 0])
attack_rows = total_rows - normal_rows
print(f"[name] : {normal_rows} normal_rows ({(normal_rows/total_rows)*100:.2f}%), {attack_rows} attack rows ({(attack_rows/total_rows)*100:.2f}%)")
'''

'# original vs normal rows\ntotal_rows = len(phy16sdf)\nnormal_rows = len(phy16sdf[phy16sdf[\'num_attacks\'] == 0])\nattack_rows = total_rows - normal_rows\nprint(f"[name] : {normal_rows} normal_rows ({(normal_rows/total_rows)*100:.2f}%), {attack_rows} attack rows ({(attack_rows/total_rows)*100:.2f}%)")\n\n# balance the phydf dataframe\nnormal_rows = phy16sdf[phy16sdf[\'num_attacks\'] == 0]\nattack_rows = phy16sdf[phy16sdf[\'num_attacks\'] == 1]\nnum_normal = len(normal_rows)\nnum_attack = len(attack_rows)\n\nnormal_rows = normal_rows.sample(n=num_attack, random_state=42)\n\nphys16df_balanced = pd.concat([normal_rows, attack_rows]).reset_index(drop=True)\n\n# new attack vs normal ratio\ntotal_rows = len(phys16df_balanced)\nnormal_rows = len(phys16df_balanced[phys16df_balanced[\'num_attacks\'] == 0])\nattack_rows = total_rows - normal_rows\nprint(f"[name] : {normal_rows} normal_rows ({(normal_rows/total_rows)*100:.2f}%), {attack_rows} attack rows ({(attack_rows/total_rows)*100:.2f}%)")\

In [24]:
'''# original vs normal rows
total_rows = len(phy10sdf)
normal_rows = len(phy10sdf[phy10sdf['num_attacks'] == 0])
attack_rows = total_rows - normal_rows
print(f"[name] : {normal_rows} normal_rows ({(normal_rows/total_rows)*100:.2f}%), {attack_rows} attack rows ({(attack_rows/total_rows)*100:.2f}%)")

# balance the phydf dataframe
normal_rows = phy10sdf[phy10sdf['num_attacks'] == 0]
attack_rows = phy10sdf[phy10sdf['num_attacks'] == 1]
num_normal = len(normal_rows)
num_attack = len(attack_rows)

normal_rows = normal_rows.sample(n=num_attack, random_state=42)

phys10df_balanced = pd.concat([normal_rows, attack_rows]).reset_index(drop=True)

# new attack vs normal ratio
total_rows = len(phys10df_balanced)
normal_rows = len(phys10df_balanced[phys10df_balanced['num_attacks'] == 0])
attack_rows = total_rows - normal_rows
print(f"[name] : {normal_rows} normal_rows ({(normal_rows/total_rows)*100:.2f}%), {attack_rows} attack rows ({(attack_rows/total_rows)*100:.2f}%)")
'''

'# original vs normal rows\ntotal_rows = len(phy10sdf)\nnormal_rows = len(phy10sdf[phy10sdf[\'num_attacks\'] == 0])\nattack_rows = total_rows - normal_rows\nprint(f"[name] : {normal_rows} normal_rows ({(normal_rows/total_rows)*100:.2f}%), {attack_rows} attack rows ({(attack_rows/total_rows)*100:.2f}%)")\n\n# balance the phydf dataframe\nnormal_rows = phy10sdf[phy10sdf[\'num_attacks\'] == 0]\nattack_rows = phy10sdf[phy10sdf[\'num_attacks\'] == 1]\nnum_normal = len(normal_rows)\nnum_attack = len(attack_rows)\n\nnormal_rows = normal_rows.sample(n=num_attack, random_state=42)\n\nphys10df_balanced = pd.concat([normal_rows, attack_rows]).reset_index(drop=True)\n\n# new attack vs normal ratio\ntotal_rows = len(phys10df_balanced)\nnormal_rows = len(phys10df_balanced[phys10df_balanced[\'num_attacks\'] == 0])\nattack_rows = total_rows - normal_rows\nprint(f"[name] : {normal_rows} normal_rows ({(normal_rows/total_rows)*100:.2f}%), {attack_rows} attack rows ({(attack_rows/total_rows)*100:.2f}%)")\

### Balance netdf

In [25]:
# original vs normal rows
total_rows = len(scada30sdf)
normal_rows = len(scada30sdf[scada30sdf['num_attacks'] == 0])
attack_rows = total_rows - normal_rows
print(f"[name] : {normal_rows} normal_rows ({(normal_rows/total_rows)*100:.2f}%), {attack_rows} attack rows ({(attack_rows/total_rows)*100:.2f}%)")

# balance the netdf dataframe
normal_rows = scada30sdf[scada30sdf['num_attacks'] == 0]
attack_rows = scada30sdf[scada30sdf['num_attacks'] == 1]
num_normal = len(normal_rows)
num_attack = len(attack_rows)

normal_rows = normal_rows.sample(n=num_attack, random_state=42)

scada30sdf_balanced = pd.concat([normal_rows, attack_rows]).reset_index(drop=True)

# new attack vs normal ratio
total_rows = len(scada30sdf_balanced)
normal_rows = len(scada30sdf_balanced[scada30sdf_balanced['num_attacks'] == 0])
attack_rows = total_rows - normal_rows
print(f"[name] : {normal_rows} normal_rows ({(normal_rows/total_rows)*100:.2f}%), {attack_rows} attack rows ({(attack_rows/total_rows)*100:.2f}%)")


[name] : 161462 normal_rows (45.77%), 191276 attack rows (54.23%)
[name] : 96802 normal_rows (50.00%), 96802 attack rows (50.00%)


In [26]:
# original vs normal rows
total_rows = len(scada16sdf)
normal_rows = len(scada16sdf[scada16sdf['num_attacks'] == 0])
attack_rows = total_rows - normal_rows
print(f"[name] : {normal_rows} normal_rows ({(normal_rows/total_rows)*100:.2f}%), {attack_rows} attack rows ({(attack_rows/total_rows)*100:.2f}%)")

# balance the netdf dataframe
normal_rows = scada16sdf[scada16sdf['num_attacks'] == 0]
attack_rows = scada16sdf[scada16sdf['num_attacks'] == 1]
num_normal = len(normal_rows)
num_attack = len(attack_rows)

normal_rows = normal_rows.sample(n=num_attack, random_state=42)

scada16sdf_balanced = pd.concat([normal_rows, attack_rows]).reset_index(drop=True)

# new attack vs normal ratio
total_rows = len(scada16sdf_balanced)
normal_rows = len(scada16sdf_balanced[scada16sdf_balanced['num_attacks'] == 0])
attack_rows = total_rows - normal_rows
print(f"[name] : {normal_rows} normal_rows ({(normal_rows/total_rows)*100:.2f}%), {attack_rows} attack rows ({(attack_rows/total_rows)*100:.2f}%)")


[name] : 161462 normal_rows (45.77%), 191276 attack rows (54.23%)
[name] : 96802 normal_rows (50.00%), 96802 attack rows (50.00%)


In [27]:
# original vs normal rows
total_rows = len(scada10sdf)
normal_rows = len(scada10sdf[scada10sdf['num_attacks'] == 0])
attack_rows = total_rows - normal_rows
print(f"[name] : {normal_rows} normal_rows ({(normal_rows/total_rows)*100:.2f}%), {attack_rows} attack rows ({(attack_rows/total_rows)*100:.2f}%)")

# balance the netdf dataframe
normal_rows = scada10sdf[scada10sdf['num_attacks'] == 0]
attack_rows = scada10sdf[scada10sdf['num_attacks'] == 1]
num_normal = len(normal_rows)
num_attack = len(attack_rows)

normal_rows = normal_rows.sample(n=num_attack, random_state=42)

scada10sdf_balanced = pd.concat([normal_rows, attack_rows]).reset_index(drop=True)

# new attack vs normal ratio
total_rows = len(scada10sdf_balanced)
normal_rows = len(scada10sdf_balanced[scada10sdf_balanced['num_attacks'] == 0])
attack_rows = total_rows - normal_rows
print(f"[name] : {normal_rows} normal_rows ({(normal_rows/total_rows)*100:.2f}%), {attack_rows} attack rows ({(attack_rows/total_rows)*100:.2f}%)")


[name] : 161462 normal_rows (45.77%), 191276 attack rows (54.23%)
[name] : 96802 normal_rows (50.00%), 96802 attack rows (50.00%)


## Classification model for the physical data

### Extract feature and target array

In [30]:
# Drop num_attacks and attack_types columns from features
X, y = phy30sdf.drop(['num_attacks', 'attack_types'], axis=1), phy30sdf['attack_types']

# Use the label encoder to transform the attack types
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
print(np.unique(y_encoded))
print(label_encoder.classes_)

# Use a one hot encoder for the rest of the categorical features
onehot_encoder = OneHotEncoder(sparse=False)
X_encoded = onehot_encoder.fit_transform(X)



[0 1 2 3 4]
['DoS' 'MITM' 'normal' 'physical fault' 'scan']


### Split the data

In [31]:
# We split the encoded data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y_encoded, random_state=1)

### Convert into DMatrix

In [32]:
# Create DMatrix for XGBoost
dtrain_clf = xgb.DMatrix(X_train, y_train, enable_categorical=True)
dtest_clf = xgb.DMatrix(X_test, y_test, enable_categorical=True)

### Define hyperparameters

In [33]:
params = {
    "objective": "multi:softprob", # multi-class classification
    "tree_method": "hist",         # use histogram-based algorithm
    "num_class": 6                 # number of classes is 5
}


### Train the model

In [34]:
n = 1000

model = xgb.train(
   params=params,     # training parameters
   dtrain=dtrain_clf, # training data
   num_boost_round=n, # number of boosting rounds
)

preds = model.predict(dtest_clf)

### Cross validate

In [35]:
results = xgb.cv(
   params, dtrain_clf,   # parameters and training data
   num_boost_round=n,   # number of boosting rounds
   nfold=5,   # number of folds for cross-validation
   metrics=["mlogloss", "auc", "merror"],  # evaluation metrics
   early_stopping_rounds=50,   # stop if no improvement after 50 rounds
   as_pandas=True,   # return results as pandas DataFrame
   verbose_eval=10.  # verbose evaluation every 10 rounds
)

/home/tbeaugh/.local/lib/python3.10/site-packages/xgboost/training.py:235: UserWarning: [00:02:33] WARNING: /workspace/src/metric/auc.cc:324: Dataset is empty, or contains only positive or negative samples.
  return self.bst.eval_set(self.watchlist, iteration, feval, output_margin)


[0]	train-mlogloss:1.17117+0.00083	train-auc:nan+nan	train-merror:0.23310+0.00028	test-mlogloss:1.18117+0.00194	test-auc:nan+nan	test-merror:0.23844+0.00122


/home/tbeaugh/.local/lib/python3.10/site-packages/xgboost/training.py:235: UserWarning: [00:02:34] WARNING: /workspace/src/metric/auc.cc:324: Dataset is empty, or contains only positive or negative samples.
  return self.bst.eval_set(self.watchlist, iteration, feval, output_margin)
/home/tbeaugh/.local/lib/python3.10/site-packages/xgboost/training.py:235: UserWarning: [00:02:49] WARNING: /workspace/src/metric/auc.cc:324: Dataset is empty, or contains only positive or negative samples.
  return self.bst.eval_set(self.watchlist, iteration, feval, output_margin)
/home/tbeaugh/.local/lib/python3.10/site-packages/xgboost/training.py:235: UserWarning: [00:03:05] WARNING: /workspace/src/metric/auc.cc:324: Dataset is empty, or contains only positive or negative samples.
  return self.bst.eval_set(self.watchlist, iteration, feval, output_margin)
/home/tbeaugh/.local/lib/python3.10/site-packages/xgboost/training.py:235: UserWarning: [00:03:16] WARNING: /workspace/src/metric/auc.cc:324: Dataset i

[10]	train-mlogloss:0.55457+0.00115	train-auc:nan+nan	train-merror:0.21342+0.00345	test-mlogloss:0.56792+0.00443	test-auc:nan+nan	test-merror:0.21747+0.00503


/home/tbeaugh/.local/lib/python3.10/site-packages/xgboost/training.py:235: UserWarning: [00:05:17] WARNING: /workspace/src/metric/auc.cc:324: Dataset is empty, or contains only positive or negative samples.
  return self.bst.eval_set(self.watchlist, iteration, feval, output_margin)
/home/tbeaugh/.local/lib/python3.10/site-packages/xgboost/training.py:235: UserWarning: [00:05:33] WARNING: /workspace/src/metric/auc.cc:324: Dataset is empty, or contains only positive or negative samples.
  return self.bst.eval_set(self.watchlist, iteration, feval, output_margin)
/home/tbeaugh/.local/lib/python3.10/site-packages/xgboost/training.py:235: UserWarning: [00:05:48] WARNING: /workspace/src/metric/auc.cc:324: Dataset is empty, or contains only positive or negative samples.
  return self.bst.eval_set(self.watchlist, iteration, feval, output_margin)
/home/tbeaugh/.local/lib/python3.10/site-packages/xgboost/training.py:235: UserWarning: [00:06:04] WARNING: /workspace/src/metric/auc.cc:324: Dataset i

...

[60]	train-mlogloss:0.28049+0.00098	train-auc:nan+nan	train-merror:0.14474+0.00128	test-mlogloss:0.30527+0.00397	test-auc:nan+nan	test-merror:0.16789+0.00440


/home/tbeaugh/.local/lib/python3.10/site-packages/xgboost/training.py:235: UserWarning: [00:17:46] WARNING: /workspace/src/metric/auc.cc:324: Dataset is empty, or contains only positive or negative samples.
  return self.bst.eval_set(self.watchlist, iteration, feval, output_margin)
/home/tbeaugh/.local/lib/python3.10/site-packages/xgboost/training.py:235: UserWarning: [00:18:00] WARNING: /workspace/src/metric/auc.cc:324: Dataset is empty, or contains only positive or negative samples.
  return self.bst.eval_set(self.watchlist, iteration, feval, output_margin)
/home/tbeaugh/.local/lib/python3.10/site-packages/xgboost/training.py:235: UserWarning: [00:18:16] WARNING: /workspace/src/metric/auc.cc:324: Dataset is empty, or contains only positive or negative samples.
  return self.bst.eval_set(self.watchlist, iteration, feval, output_margin)
/home/tbeaugh/.local/lib/python3.10/site-packages/xgboost/training.py:235: UserWarning: [00:18:31] WARNING: /workspace/src/metric/auc.cc:324: Dataset i

[70]	train-mlogloss:0.26689+0.00096	train-auc:nan+nan	train-merror:0.14191+0.00079	test-mlogloss:0.29425+0.00399	test-auc:nan+nan	test-merror:0.17223+0.00316


/home/tbeaugh/.local/lib/python3.10/site-packages/xgboost/training.py:235: UserWarning: [00:19:56] WARNING: /workspace/src/metric/auc.cc:324: Dataset is empty, or contains only positive or negative samples.
  return self.bst.eval_set(self.watchlist, iteration, feval, output_margin)
/home/tbeaugh/.local/lib/python3.10/site-packages/xgboost/training.py:235: UserWarning: [00:20:08] WARNING: /workspace/src/metric/auc.cc:324: Dataset is empty, or contains only positive or negative samples.
  return self.bst.eval_set(self.watchlist, iteration, feval, output_margin)
/home/tbeaugh/.local/lib/python3.10/site-packages/xgboost/training.py:235: UserWarning: [00:20:09] WARNING: /workspace/src/metric/auc.cc:324: Dataset is empty, or contains only positive or negative samples.
  return self.bst.eval_set(self.watchlist, iteration, feval, output_margin)
/home/tbeaugh/.local/lib/python3.10/site-packages/xgboost/training.py:235: UserWarning: [00:20:24] WARNING: /workspace/src/metric/auc.cc:324: Dataset i

[74]	train-mlogloss:0.26239+0.00092	train-auc:nan+nan	train-merror:0.14090+0.00185	test-mlogloss:0.29099+0.00383	test-auc:nan+nan	test-merror:0.17376+0.00376


### Calculate performace metrics 

In [36]:
# Find the target labels from the predictions
y_pred = np.asarray([np.argmax(line) for line in preds])

# Calculate precision, recall, f1-score, and confusion matrix
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')
cm = confusion_matrix(y_test, y_pred)

# Print the results
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-score: {f1:.3f}")
print("Confusion Matrix:\n", cm)

Precision: 0.468
Recall: 0.490
F1-score: 0.478
Confusion Matrix:
 [[  82    0  108    0    0]
 [   0  286  138    0    0]
 [ 133  171 2702  212  103]
 [   0    0  184  136    0]
 [   0    0  103    0   12]]


## Classification model for physical 16s bucket

### Extract feature and target array

In [ ]:
# Drop num_attacks and attack_types columns from features
X, y = phys16df_balanced.drop(['num_attacks', 'attack_types'], axis=1), phys16df_balanced['attack_types']

# Use the label encoder to transform the attack types
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Use a one hot encoder for the rest of the categorical features
onehot_encoder = OneHotEncoder(sparse=False)
X_encoded = onehot_encoder.fit_transform(X)

### Split the data

In [ ]:
# We split the encoded data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y_encoded, random_state=1)

### Convert into dmatrix

In [ ]:
# Create DMatrix for XGBoost
dtrain_clf = xgb.DMatrix(X_train, y_train, enable_categorical=True)
dtest_clf = xgb.DMatrix(X_test, y_test, enable_categorical=True)

### Define hyperparameters

In [ ]:
params = {
    "objective": "multi:softprob", # multi-class classification
    "tree_method": "hist",         # use histogram-based algorithm
    "num_class": 6                 # number of classes is 5
}


### Train the model

In [ ]:
n = 1000

model = xgb.train(
   params=params,     # training parameters
   dtrain=dtrain_clf, # training data
   num_boost_round=n, # number of boosting rounds
)

preds = model.predict(dtest_clf)

### Cross validate

In [ ]:
results = xgb.cv(
   params, dtrain_clf,   # parameters and training data
   num_boost_round=n,   # number of boosting rounds
   nfold=5,   # number of folds for cross-validation
   metrics=["mlogloss", "auc", "merror"],  # evaluation metrics
   early_stopping_rounds=50,   # stop if no improvement after 50 rounds
   as_pandas=True,   # return results as pandas DataFrame
   verbose_eval=10.  # verbose evaluation every 10 rounds
)

### Calculate performance metrics

In [ ]:
# Find the target labels from the predictions
y_pred = np.asarray([np.argmax(line) for line in preds])

# Calculate precision, recall, f1-score, and confusion matrix
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')
cm = confusion_matrix(y_test, y_pred)

# Print the results
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-score: {f1:.3f}")
print("Confusion Matrix:\n", cm)

## Classification model for network data

### Extract feature and target array

In [ ]:
# Drop num_attacks and attack_types columns from features
X, y = netdf_balanced.drop(['num_attacks', 'attack_types'], axis=1), netdf_balanced['attack_types']

# Use the label encoder to transform the attack types
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Use a one hot encoder for the rest of the categorical features
onehot_encoder = OneHotEncoder(sparse=False)
X_encoded = onehot_encoder.fit_transform(X)

### Split the data

In [ ]:
# We split the encoded data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y_encoded, random_state=1)

### Convert into DMatrix

In [ ]:
# Create DMatrix for XGBoost
dtrain_clf = xgb.DMatrix(X_train, y_train, enable_categorical=True)
dtest_clf = xgb.DMatrix(X_test, y_test, enable_categorical=True)


### Define hyperparameters

In [ ]:
params = {
    "objective": "multi:softprob", # multi-class classification
    "tree_method": "hist",         # use histogram-based algorithm
    "num_class": 6                 # number of classes is 5
}


### Train the model

In [ ]:
n = 1000

model = xgb.train(
   params=params,     # training parameters
   dtrain=dtrain_clf, # training data
   num_boost_round=n, # number of boosting rounds
)

preds = model.predict(dtest_clf)

### Cross validate

In [ ]:
results = xgb.cv(
   params, dtrain_clf,   # parameters and training data
   num_boost_round=n,   # number of boosting rounds
   nfold=5,   # number of folds for cross-validation
   metrics=["mlogloss", "auc", "merror"],  # evaluation metrics
   early_stopping_rounds=50,   # stop if no improvement after 50 rounds
   as_pandas=True,   # return results as pandas DataFrame
   verbose_eval=10.  # verbose evaluation every 10 rounds
)

### Calculate performace metrics

In [ ]:
# Find the target labels from the predictions
y_pred = np.asarray([np.argmax(line) for line in preds])

# Calculate precision, recall, f1-score, and confusion matrix
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')
cm = confusion_matrix(y_test, y_pred)

# Print the results
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-score: {f1:.3f}")
print("Confusion Matrix:\n", cm)